Bounding functions
==================

**Author:** Rafael



## Sintaxis



Hemos visto como `zip` es un generador que dadas dos listas genera una lista de parejas tomando un elemento de cada lista:



In [1]:
frutas = ["limón", "manzana", "guayaba"]
colores = ["verde", "rojo", "amarillo"]
precio = [20, 40, 30]
pares = zip(frutas, colores, precio)
pares

In [1]:
next(pares)

In [1]:
list(pares)

Sin embargo, `zip` puede recibir muchos argumentes, y si los argumentos son parejas, produce "listas transversales".



In [1]:
parejas = [('limón', 'verde', 20), ('manzana', 'rojo', 40), ('guayaba', 'amarillo', 30)]
list(zip(*parejas))

## Definiciones



Una manera más sofisticada de hacer podado en el árbol de estados es por medio de *funciones acotadoras*. Este concepto se aplica a problemas de optimización donde se generan soluciones factibles por *backtracking*, como por ejemplo, el problema de la mochila.

Dada una solución parcial factible, $X=[x_{0},x_{1},\ldots,x_{l-1}]$, sea $P(X)$ la ganancia máxima entre todas las soluciones factibles que sean descendientes de $X$ en el árbol de estados.

En particular $P([\ ])$ es la ganancia máxima en nuestro problema. En general, $P(X)$ es difícil de calcular.

Una *función acotadora* es una función $B$ con dominio los vértices del árbol de estados (o sea, el conjunto de soluciones parciales) y valores en $\mathbb{R}$, tal que para toda solución parcial factible $X$ se tiene que $B(X)\geq P(X)$.

Supongamos entonces que tenemos una función acotadora $B$ tal que si $OptP$ es la ganancia óptima actual (la mejor ganancia obtenida hasta el momento) y $X$ es una solución parcial tal que $B(X)\leq OptP$. Entonces tendríamos:

\begin{equation}
\label{exa:1}
P(X)\leq B(X)\leq OptP
\end{equation}

Lo cual implica que ningún descendiente de $X$ va a mejorar la ganancia óptima, por lo que podemos podar el árbol de todos los descendientes de la solución parcial $X$.

Debemos pensar que $B(X)$ aproxima a $P(X)$ por arriba. Buscamos funciones acotadoras tales que:

1.  sean fáciles de calcular
2.  sus valores sean cercanos a los de $P(X)$.

Estas dos condiciones compiten entre sí. Por ejemplo, $P(X)$ misma es una función acotadora, pero es difícil de calcular.



## Problema de la mochila racional



Consideremos el siguiente problema (RationalKnapsack)

Dadas

-   ganancias $p_{0},p_{1},\ldots,p_{n-1}$,
-   pesos $w_{0},w_{1},\ldots,w_{n-1}$,
-   capacidad $M$

encontrar el máximo valor de $\sum_{i=0}^{n-1}p_{i}x_{i}$ sujeto a $\sum_{i=0}^{i=n-1}w_{i}x_{i}\leq M$, con $x_{i}\in \mathbb{Q}\cap [0,1]$.

Debido a la condición de que las variables $x_{i}$ estén en el intervalo $[0, 1]$ este no es un problema usual de programación lineal. Sin embargo el siguiente código lo resuelve.



In [1]:
from dataclasses import dataclass

@dataclass
class KnapsackInstance:
    weights: list[int]
    profits: list[int]
    capacity: int
    
    def __post_init__(self):
        # Combine profits and weights to sort them by density (profit/weight ratio)
        sorted_items = sorted(
            zip(self.profits, self.weights),
            key=lambda item: item[0] / item[1],
            reverse=True
        )
        self.profits, self.weights = zip(*sorted_items)
        # Convert zip objects back to lists
        self.profits = list(self.profits)
        self.weights = list(self.weights)

def fractional_knapsack(instance):
    total_profit = 0.0
    current_weight_in_bag = 0.0
    capacity_limit = instance.capacity
    number_of_items = len(instance.weights)
    
    # fractions_taken stores what percentage of each item we put in the bag
    fractions_taken = [0.0] * number_of_items
    
    for i in range(number_of_items):
        # If the bag is full, we stop
        if current_weight_in_bag >= capacity_limit:
            break
            
        weight = instance.weights[i]
        profit = instance.profits[i]
        
        # Check if the whole item fits
        if current_weight_in_bag + weight <= capacity_limit:
            fractions_taken[i] = 1.0
            current_weight_in_bag += weight
            total_profit += profit
        else:
            # Take only a fraction to fill the remaining space
            remaining_space = capacity_limit - current_weight_in_bag
            fraction = remaining_space / weight
            fractions_taken[i] = fraction
            total_profit += fraction * profit
            
    return total_profit, fractions_taken

# --- Example Usage ---
instance = KnapsackInstance(
    weights=[20, 10, 30], 
    profits=[100, 60, 120], 
    capacity=50
)

max_value, breakdown = fractional_knapsack(instance)

print(f"Items sorted by weight: {instance.weights}")
print(f"Total value in bag: {max_value}")
print(f"Proportion taken per item: {breakdown}")

## Función acotadora para Knapsack



Usaremos la función $RKnap$ que resuelve el problema RationalKnapsack con lista de ganancias $[p_{0},p_{1},\ldots,p_{n-1}]$, lista de pesos $[w_{0},w_{1},\ldots,w_{n-1}]$ y capacidad $M$. Dada una solución parcial factible $[x_{0},x_{1},\ldots,x_{l-1}]$, definimos:

\begin{equation}
\label{exa:2}
B(X)=\sum_{i=0}^{l-1}p_{i}x_{i}+RKnap([p_{l},\ldots,p_{n}],[w_{l},\ldots,w_{n}], M-\sum_{i=0}^{l-1}w_{i}x_{i})
\end{equation}

Esta es una función fácil de calcular que cumple $P(X)\leq B(X)$.



In [1]:
def get_fractional_upper_bound(instance, start_index, remaining_capacity):
    """Calculates the RKNAP bound (B) for the remaining capacity."""
    total_profit = 0.0
    current_capacity = remaining_capacity
    
    for i in range(start_index, len(instance.weights)):
        if current_capacity <= 0:
            break
        
        weight = instance.weights[i]
        profit = instance.profits[i]
        
        if weight <= current_capacity:
            current_capacity -= weight
            total_profit += profit
        else:
            total_profit += profit * (current_capacity / weight)
            current_capacity = 0
            
    return total_profit


def knapsack_with_bound(instance):
    best = {"profit": 0, "weight": 0, "selection": []}

    def solve(idx, current, current_weight, current_profit):
        if idx == len(instance.weights):
            if current_profit > best["profit"]:
                best["profit"] = current_profit
                best["weight"] = current_weight
                best["selection"] = current[:]
            return

        # Calculate the Upper Bound B for this level
        remaining_cap = instance.capacity - current_weight
        upper_bound = current_profit + get_fractional_upper_bound(
            instance, idx, remaining_cap
        )

        # Pruning logic: If the bound isn't better than best, we only explore choice 0
        if upper_bound <= best["profit"]:
            choices = [0]
        else:
            # Standard capacity check for the 'Take' branch
            if current_weight + instance.weights[idx] <= instance.capacity:
                choices = [1, 0]
            else:
                choices = [0]

        for choice in choices:
            current.append(choice)
            
            # Calculate weight and profit to pass down
            item_weight = instance.weights[idx] if choice == 1 else 0
            item_profit = instance.profits[idx] if choice == 1 else 0
            
            solve(idx + 1, current, current_weight + item_weight, current_profit + item_profit)
            
            current.pop()

    # Start recursion with initial profit of 0
    solve(0, [], 0, 0)
    return best

# --- Test ---
prob = KnapsackInstance([10, 20, 30], [60, 100, 120], 50)
knapsack_with_bound(prob)

## Comparación



Recordemos el código con el podado simple:



In [1]:
def knapsack_solver_prune(instance):
    best = {"profit": 0, "weight": 0, "selection": []}

    def solve(idx, current, current_weight):
        if idx == len(instance.weights):
            total_profit = sum(p * s for p, s in zip(instance.profits, current))
            if total_profit > best["profit"]:
                best["profit"] = total_profit
                best["weight"] = current_weight
                best["selection"] = current[:]
            return

        # Check if adding the next item (choice 1) exceeds capacity
        if current_weight + instance.weights[idx] <= instance.capacity:
            choices = [1, 0]
        else:
            choices = [0]

        for choice in choices:
            current.append(choice)
            # Update current_weight based on the choice made
            new_weight = current_weight + (instance.weights[idx] if choice == 1 else 0)
            solve(idx + 1, current, new_weight)
            current.pop()

    solve(0, [], 0)
    return best

In [1]:
weights = [15, 12, 10, 8, 25, 30, 5, 18, 22, 14, 11, 9, 20, 13, 7, 24, 16, 19, 21, 6, 17, 23, 4, 28, 12, 10, 15, 8, 26, 19]
profits = [20, 15, 12, 10, 30, 35, 8, 22, 25, 18, 14, 11, 25, 17, 9, 28, 20, 23, 26, 8, 21, 27, 5, 32, 16, 13, 19, 11, 31, 24]
capacity = 120
prob = KnapsackInstance(weights, profits, capacity)

In [1]:
len(prob.weights)

In [1]:
knapsack_solver_prune(prob)

In [1]:
knapsack_with_bound(prob)